In [1]:
import kagglehub
import pandas as pd
import os
import ast
import html
import re
import string
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk 
nltk.download('punkt_tab')
nltk.download('stopwords')

/Users/jelenalazovic/ml/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jelenalazovic/ml/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jelenalazovic/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jelenalazovic/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Ucitavanje sirovih podataka

In [2]:
path = kagglehub.dataset_download("shuyangli94/foodcom-recipes-with-search-terms-and-tags")
csv_path = os.path.join(path, 'recipes_w_search_terms.csv')
df = pd.read_csv(csv_path)

def get_cuisine(x):
    if not isinstance(x, str):
        return None
    x_lower = x.lower()
    if 'italian' in x_lower:
        return 'Italian'
    if 'indian' in x_lower:
        return 'Indian'
    return None

italian_indian_df = df[df['search_terms'].apply(lambda x: isinstance(x, str) and ('italian' in x.lower() or 'indian' in x.lower()))].copy()
italian_indian_df['cuisine'] = italian_indian_df['search_terms'].apply(get_cuisine)
final_df = italian_indian_df[['name', 'steps', 'cuisine']].reset_index(drop=True)
final_df['steps'] = final_df['steps'].apply(lambda x: ' '.join(ast.literal_eval(x)) if isinstance(x, str) else x)
final_df['name'] = final_df['name'].str.lower()
final_df['steps'] = final_df['steps'].str.lower()

italian_sample = final_df[final_df['cuisine'] == 'Italian'].sample(n=10000, random_state=42)
indian_all = final_df[final_df['cuisine'] == 'Indian']

final_df = pd.concat([italian_sample, indian_all], ignore_index=True)
final_df.to_csv('./data/recipes_raw.csv')

Osnovne provere

In [3]:
print('Number of null rows\n', final_df.isna().sum())
print('Number of duplicated rows', final_df.duplicated().sum())
df_no_dub = final_df.drop_duplicates().reset_index(drop=True)

Number of null rows
 name       0
steps      0
cuisine    0
dtype: int64
Number of duplicated rows 10


Čišćenje teksta (HTML entiteti, nevidljivi/kontrolni karakteri)

In [4]:
# zero-width space (200b), line/paragraph separator (2028/2029), BOM (feff), nbsp (a0)
_invisible = [0x200b, 0x2028, 0x2029, 0xfeff, 0xa0]
INVISIBLE_CHARS = re.compile('[' + ''.join(chr(c) for c in _invisible) + ']')
CONTROL_CHARS = re.compile('[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = html.unescape(text)  # &amp; -> &, &rsquo; -> ’, &eacute; -> é, ...
    text = unicodedata.normalize('NFKC', text)
    text = INVISIBLE_CHARS.sub(' ', text)
    text = CONTROL_CHARS.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean = df_no_dub.copy()
df_clean['name'] = df_clean['name'].apply(clean_text)
df_clean['steps'] = df_clean['steps'].apply(clean_text)

Tokenizacija i priprema naslova recepata (name_tokens)

In [5]:
df_clean['name_tokens'] = df_clean['name'].apply(word_tokenize)
df_clean['steps_tokens'] = df_clean['steps'].apply(word_tokenize)
df_clean.to_csv('./data/recipes_tokenized.csv')

In [6]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

def remove_punctuation(tokens):
    return [token for token in tokens if token not in string.punctuation]

df_clean['name_tokens'] = df_clean['name_tokens'].apply(remove_stopwords)
df_clean['name_tokens'] = df_clean['name_tokens'].apply(remove_punctuation)
# steps_tokens ostaje netaknut ovde — to je seq2seq (RNN_enc/RNN_dec) target.
# Stop reči/interpunkcija se kasnije uklanjaju iz kopije za BoW klasifikator (steps_tokens_bow).

Statisticka analiza podataka

In [ ]:
df_clean['number_of_tokens'] = df_clean['steps_tokens'].apply(len)
print(df_clean['number_of_tokens'].describe())

max_len = int(df_clean['number_of_tokens'].quantile(0.95))
df_clean_no_outliers = df_clean[
    df_clean['number_of_tokens'].between(13, max_len - 1)
].copy()
print(df_clean_no_outliers['cuisine'].value_counts())

count    16546.000000
mean       144.088541
std         98.043552
min          2.000000
25%         81.000000
50%        123.000000
75%        181.000000
max       1716.000000
Name: number_of_tokens, dtype: float64
cuisine
Italian    9376
Indian     6245
Name: count, dtype: int64


Dodatno čišćenje steps_tokens (brojevi, mere, artefakti, razlomci, crtice)

In [8]:
MEASUREMENT_WORDS = {
    "cup", "cups",
    "tbsp", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",
    "qt", "quart", "quarts",
    "pint", "pints",
    "g", "kg", "mg",
    "ml", "l",
    "inch", "inches",
    "degree", "degrees",
    "minute", "minutes",
    "hour", "hours"
}

def remove_numbers_measurements(tokens):
    cleaned = []
    for token in tokens:
        token = token.lower()
        # remove pure numbers and fractions
        if re.fullmatch(r"[\d¼½¾⁄/.-]+", token):
            continue
        if token in MEASUREMENT_WORDS:
            continue
        cleaned.append(token)
    return cleaned

ARTIFACTS = {
    "'s", "'re", "'ve", "'ll", "'d", "'m", "n't", "--"
}

def remove_artifacts(tokens):
    return [t for t in tokens if t not in ARTIFACTS]

def normalize_unicode_fractions(tokens):
    replacements = {"½": "1/2", "¼": "1/4", "¾": "3/4", "⁄": "/"}
    normalized = []
    for token in tokens:
        for old, new in replacements.items():
            token = token.replace(old, new)
        normalized.append(token)
    return normalized

def split_hyphenated(tokens):
    output = []
    for token in tokens:
        output.extend(token.replace("-", " ").split())
    return output

df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(remove_numbers_measurements)
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(remove_artifacts)
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(normalize_unicode_fractions)
df_clean_no_outliers["steps_tokens"] = df_clean_no_outliers["steps_tokens"].apply(split_hyphenated)

Grananje: BoW obeležja za sadržajni klasifikator (steps_tokens_bow)

steps_tokens ostaje netaknut kao seq2seq target za RNN_enc/RNN_dec.
steps_tokens_bow je kopija sa uklonjenim stop rečima i interpunkcijom, namenjena samo BoW/multi-task klasifikatoru.

In [9]:
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens"].apply(remove_stopwords)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_punctuation)

df_clean_no_outliers.to_csv('./data/recipes_final.csv')

Train-test-val split 

In [10]:
from sklearn import model_selection

df = pd.read_csv('./data/recipes_final.csv', index_col=0)
df['steps_tokens'] = df['steps_tokens'].apply(ast.literal_eval)
df['steps_tokens_bow'] = df['steps_tokens_bow'].apply(ast.literal_eval)

X = df[['steps_tokens', 'steps_tokens_bow']]
y = df['cuisine']

X_train, X_temp, y_train, y_temp = model_selection.train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = model_selection.train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [11]:
train_df = X_train.join(y_train)
val_df = X_val.join(y_val)
test_df = X_test.join(y_test)

train_df.to_csv('./data/recipes_train.csv')
val_df.to_csv('./data/recipes_val.csv')
test_df.to_csv('./data/recipes_test.csv')

print(train_df.shape, val_df.shape, test_df.shape)

(10934, 3) (2343, 3) (2344, 3)


Word2Vec

In [12]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=train_df['steps_tokens'],
    vector_size=150,
    window=5,
    min_count=2,
    workers=4,
    epochs=15
)

model.wv.save('./data/models/word2vec.wordvectors')

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


BoW

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
import joblib

def identity_analyzer(tokens):
    return tokens

vectorizer = CountVectorizer(analyzer=identity_analyzer)

X_train_bow = vectorizer.fit_transform(train_df['steps_tokens_bow'])
X_val_bow = vectorizer.transform(val_df['steps_tokens_bow'])
X_test_bow = vectorizer.transform(test_df['steps_tokens_bow'])

joblib.dump(vectorizer, './data/models/bow_vectorizer.joblib')

['./data/models/bow_vectorizer.joblib']